In [1]:
import torch
import torch.nn as nn

# 历史输入：总用户数、预约人数、播出时间、推送方式
input_raw = torch.tensor([
    [30000, 1800, 14, 1],
    [40000, 2000, 16, 1],
    [50000, 3200, 20, 2],
    [60000, 2500, 12, 0],
    [70000, 3000, 15, 0],
    [80000, 4200, 18, 1],
    [100000, 6000, 20, 2],
    [120000, 7500, 21, 2],
    [150000, 9000, 19, 2],
    [200000, 15000, 22, 2]
], dtype=torch.float32)

# 历史输出：在线人数峰值
output = torch.tensor([4100, 4300, 9000, 2800, 3600, 9100, 15800, 19200, 22500, 38000], dtype=torch.float32).reshape(-1, 1)

# 标准化输入
mean = input_raw.mean(dim=0)
std = input_raw.std(dim=0)
std = torch.where(std == 0, torch.ones_like(std), std)
input_zscore = (input_raw - mean) / std

print("Mean:", mean)
print("Std:", std)
print("Normalized Input:\n", input_zscore)

model = nn.Linear(in_features=4, out_features=1)
losser = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)

for epoch in range(20000):
    pred = model(input_zscore)
    loss = losser(pred, output)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f'Epoch {epoch}: Loss = {loss.item():.2f}')

# 推理函数
def inference(raw_input, model, mean, std):
    normalized_input = (raw_input - mean) / std
    with torch.no_grad():
        pred = model(normalized_input)
    return pred.squeeze().item()

# 测试已有样本
true_values = output.flatten().tolist()
for i in range(len(input_raw)):
    raw_x = input_raw[i:i+1]  # 保持 batch 维度
    true_y = true_values[i]
    pred_y = inference(raw_x, model, mean, std)
    print(f"样本 {i+1:2d} | 真实值: {true_y:6.0f} | 预测值: {pred_y:8.1f} | 误差: {pred_y - true_y:8.1f}")

# 预测新样本
new_sample = torch.tensor([[180000, 12000, 20, 2]], dtype=torch.float32)
pred_new = inference(new_sample, model, mean, std)
print(f"输入: 总用户={int(new_sample[0,0])}, 预约={int(new_sample[0,1])}, 时间={int(new_sample[0,2])}点, 推送={int(new_sample[0,3])}")
print(f"预测最高在线人数: {pred_new:.0f}")

Mean: tensor([9.0000e+04, 5.4200e+03, 1.7700e+01, 1.3000e+00])
Std: tensor([5.3541e+04, 4.1480e+03, 3.3015e+00, 8.2327e-01])
Normalized Input:
 tensor([[-1.1206, -0.8727, -1.1207, -0.3644],
        [-0.9339, -0.8245, -0.5149, -0.3644],
        [-0.7471, -0.5352,  0.6966,  0.8503],
        [-0.5603, -0.7039, -1.7265, -1.5791],
        [-0.3735, -0.5834, -0.8178, -1.5791],
        [-0.1868, -0.2941,  0.0909, -0.3644],
        [ 0.1868,  0.1398,  0.6966,  0.8503],
        [ 0.5603,  0.5014,  0.9995,  0.8503],
        [ 1.1206,  0.8631,  0.3938,  0.8503],
        [ 2.0545,  2.3095,  1.3024,  0.8503]])
Epoch 0: Loss = 278825408.00
Epoch 10: Loss = 148618592.00
Epoch 20: Loss = 88776872.00
Epoch 30: Loss = 56870368.00
Epoch 40: Loss = 37917152.00
Epoch 50: Loss = 25915292.00
Epoch 60: Loss = 18055292.00
Epoch 70: Loss = 12818666.00
Epoch 80: Loss = 9297160.00
Epoch 90: Loss = 6914804.00
Epoch 100: Loss = 5295127.50
Epoch 110: Loss = 4188428.50
Epoch 120: Loss = 3427846.50
Epoch 130: Loss = 2